# **Pre-training & IFT**



**Goal:** Understand the *mechanics* behind the theory:
- how inputs are structured
- why base models fail at instruction
- how Instruction Fine-Tuning (IFT) fixes this.

## **Prerequisities & Setup**
We will use the Hugging Face `transformers` ecosystem.

In [ ]:
!pip install -q transformers datasets torch scikit-learn

In [ ]:
import torch

# Import core Hugging Face Transformers utilities.
# These classes cover the three major LLM training/inference paradigms:
# 1) Masked Language Modeling (BERT-style)
# 2) Causal Language Modeling (GPT-style)
# 3) Sequence-to-Sequence Modeling (T5-style)

from transformers import (
    AutoTokenizer,          # Automatically loads the correct tokenizer for a given model
                            # (handles tokenization, padding, truncation, vocab mapping)

    AutoModelForMaskedLM,   # Loads models trained with Masked Language Modeling (MLM)
                            # Used by BERT-like models for tasks such as
                            # fill-in-the-blank, classification heads, and embeddings

    BertTokenizer,          # Tokenizer specifically designed for BERT
                            # Uses WordPiece tokenization and BERT’s fixed vocabulary
                            # Required when working directly with BERT models (e.g., NSP, MLM)

    BertForNextSentencePrediction,  # BERT model with an added Next Sentence Prediction (NSP) head
                                    # Takes two sentences as input and predicts whether
                                    # sentence B logically follows sentence A
                                    # Used in BERT pre-training and tasks like coherence checking

    AutoModelForCausalLM,   # Loads autoregressive (left-to-right) language models
                            # Trained to predict the next token
                            # Used by GPT-style models for text generation and chat

    AutoModelForSeq2SeqLM,  # Loads encoder-decoder models for sequence-to-sequence tasks
                            # Used by T5/BART-style models for translation, summarization, QA

    DataCollatorForSeq2Seq, # Prepares batches dynamically during training for seq2seq models
                            # Handles padding, label shifting, and decoder input creation

    Seq2SeqTrainingArguments, # Configuration object for seq2seq training
                              # Defines batch size, learning rate, epochs, logging, checkpoints

    Seq2SeqTrainer,        # High-level training loop for encoder-decoder models
                            # Abstracts forward pass, loss computation, and evaluation

    pipeline               # High-level inference API for quick demos
                            # Enables tasks like text-generation, summarization, QA in one line
)

from datasets import Dataset
import pandas as pd
import gc #collects garbage and helps to clean up memory

In [ ]:
def clean_memory():
    """Utility to free up GPU/RAM for the next section."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch Version: 2.9.0+cu126
CUDA Available: True


# **Pre-training Architectures & Objectives**



We basically identifed three main architectures:
1. **Encoders (e.g., BERT):** Bidirectional context, good for understanding/classification.
2. **Encoder-Decoders (e.g., BART, T5):** Good for sequence-to-sequence tasks like translation/summarization.
3. **Decoders (e.g., GPT, Llama):** Autoregressive, good for generation.


### **The Encoder: BERT & Masked Language Modeling (MLM)**


**Theory:**
BERT uses **Masked Language Modeling (MLM)**. It masks $k\%$ (usually 15%) of the input tokens and tries to predict them based on *bidirectional* context (left and right).

* **Bidirectional Context:** "I am **[MASK]** to school" (Model sees  before and after of the mask).
* **Input Representation:** Sum of Token Embeddings + Segment Embeddings + Position Embeddings.
* No human intervention needed.

Model Used: https://huggingface.co/google-bert/bert-base-uncased

*We rarely pre-train BERT from scratch. You download a pre-trained checkpoint and fine-tune it for classification (NER, Sentiment) or use it for embedding extraction.*

In [ ]:
# We use 'bert-base-uncased', the classic encoder model
# Load the tokenizer and model
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
model_bert = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
print(tokenizer_bert)
print(model_bert)

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding

In [ ]:
# 1. Creating a raw input sentence with a mask
text_masked = "I am going to [MASK] today"

# 2. Tokenize
# 'return_tensors="pt"' converts lists directly to PyTorch tensors for the GPU
inputs = tokenizer_bert(text_masked, return_tensors="pt")


print("\n\n\n")

# 3. Inspect Input IDs to see the [MASK] token ID
mask_token_id = tokenizer_bert.mask_token_id
print(f"Input Text: {text_masked}")
print(f"Token IDs: {inputs['input_ids']}")
print(f"Mask Token ID: {mask_token_id}")





Input Text: I am going to [MASK] today
Token IDs: tensor([[ 101, 1045, 2572, 2183, 2000,  103, 2651,  102]])
Mask Token ID: 103


Guess? What is 101 and 102 here?

In [ ]:

# 4. Forward Pass (Inference)
with torch.no_grad():
    logits = model_bert(**inputs).logits

print(logits, "\n")

# 5. Retrieve predictions for the masked position
# Find the index where the mask token is located
mask_token_index = (inputs.input_ids == mask_token_id)[0].nonzero(as_tuple=True)[0]

# Get the logits for that specific token
predicted_token_id = logits[0, mask_token_index].argmax(axis=-1)
print(predicted_token_id)

predicted_token = tokenizer_bert.decode(predicted_token_id)

print(f"\n--- BERT Prediction ---")
print(f"Context: {text_masked}")
print(f"Predicted Word: '{predicted_token}'")



tensor([[[ -6.5495,  -6.4923,  -6.5326,  ...,  -5.9638,  -5.6891,  -4.0523],
         [-13.4865, -13.3951, -13.3971,  ..., -11.2246,  -9.4069, -10.9768],
         [-14.7498, -14.6224, -14.9019,  ..., -11.9540, -11.7074,  -8.6409],
         ...,
         [ -6.8451,  -6.9206,  -6.9619,  ...,  -6.5181,  -7.2667,  -5.6149],
         [-10.1398, -10.2481, -10.1501,  ...,  -8.0766,  -8.1012,  -7.8304],
         [-10.4912, -10.2839, -10.4671,  ...,  -7.9094,  -8.2507,  -7.5618]]]) 

tensor([2793])

--- BERT Prediction ---
Context: I am going to [MASK] today
Predicted Word: 'bed'


In [ ]:
# Clean up
del tokenizer_bert, model_bert, inputs, logits
clean_memory()

In [ ]:
def predict_next_sentence(sentence_a, sentence_b):
    # Note: If you have a fine-tuned model from the previous block, load that instead of 'bert-base-uncased'
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased')

    # Tokenize the pair
    # encode_plus handles adding [CLS], [SEP], and segment_ids (0 for sent_a, 1 for sent_b) automatically
    encoding = tokenizer.encode_plus(
        sentence_a,
        sentence_b,
        return_tensors='pt'  # Return PyTorch tensors
    )

    # Run Inference
    # The model expects 'input_ids' and 'token_type_ids' (segment IDs)
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits

    # 4. Interpret Results
    # The model outputs logits for 2 classes:
    # Class 0: "IsNext" (Sentence B follows Sentence A)
    # Class 1: "NotNext" (Sentence B is random/unrelated)
    probabilities = torch.softmax(logits, dim=1)
    prediction = torch.argmax(probabilities, dim=1).item()

    print(f"Sentence A: {sentence_a}")
    print(f"Sentence B: {sentence_b}")
    print(f"Is Next Sentence? {'YES' if prediction == 0 else 'NO'}")
    print(f"Scores (IsNext vs NotNext): {probabilities[0].tolist()}\n")

In [ ]:
# --- usage ---

# Example 1: True pair
seq_A = "I am fat."
seq_B = "I eat a lot."
predict_next_sentence(seq_A, seq_B)

# Example 2: False/Random pair
seq_A = "I am fat."
seq_B = "Abu Dhabi also has an AQI of 119."
predict_next_sentence(seq_A, seq_B)

Sentence A: I am fat.
Sentence B: I eat a lot.
Is Next Sentence? YES
Scores (IsNext vs NotNext): [0.9999885559082031, 1.1418412213970441e-05]

Sentence A: I am fat.
Sentence B: Abu Dhabi also has an AQI of 119.
Is Next Sentence? NO
Scores (IsNext vs NotNext): [0.0007593680638819933, 0.9992406368255615]



In [ ]:
# Clean up
del tokenizer_bert, model_bert, inputs, logits
clean_memory()

NameError: name 'tokenizer_bert' is not defined

### **The Decoder: GPT/Llama & Next Token Prediction**


**Theory:**
Decoder-only models are **Autoregressive**. =They calculate probability based on *previous* history only.
$$P(X) = \prod_{i=1}^{I} P(x_i | x_1, ..., x_{i-1})$$

**Constraint:** They cannot "see" the future words. This makes them excellent generators but potentially weaker at bidirectional understanding compared to BERT.

Model used: https://huggingface.co/openai-community/gpt2

In [ ]:
# We use 'gpt2' (small) to demonstrate the decoder architecture for Next Token Prediction

tokenizer_gpt = AutoTokenizer.from_pretrained("gpt2")
model_gpt = AutoModelForCausalLM.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
print(tokenizer_gpt)
print(model_gpt)

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_aff

In [ ]:
text_prefix = "The Prime Minister of India will be"

inputs = tokenizer_gpt(text_prefix, return_tensors="pt")

# Generate text
# do_sample=True adds randomness (temperature) vs greedy decoding
output = model_gpt.generate(**inputs, max_new_tokens=10, do_sample=False)

generated_text = tokenizer_gpt.decode(output[0], skip_special_tokens=True)


print("\n\n\n")
print(f"--- GPT-2 Generation ---")
print(f"Input Prefix: '{text_prefix}'")
print(f"Completion: '...{generated_text[len(text_prefix):]}'")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.






--- GPT-2 Generation ---
Input Prefix: 'The Prime Minister of India will be'
Completion: '... the first to speak out against the government's decision'


In [ ]:
# Clean up
del tokenizer_gpt, model_gpt, inputs, output
clean_memory()

# **The Alignment Problem (Pre-training Failures)**

**Theory:**
Pre-trained models (Base Models) often fail at **zero-shot instruction following**.
Why? Their training objective was "predict the next word," not "follow instructions".

*When asked to "Write a poem...", a base model might just continue the sentence or repeat the prompt because that's a statistically likely continuation in internet text.*

Model used: https://huggingface.co/docs/transformers/en/model_doc/gpt2

In [ ]:
# We use GPT-2 again. It is a "Base Model" (not instruction tuned like ChatGPT).

tokenizer_base = AutoTokenizer.from_pretrained("gpt2")
model_base = AutoModelForCausalLM.from_pretrained("gpt2")

# An instruction that requires following a constraint
instruction_prompt = "Instruction: What’s the best way to lose weight quickly?. \nResponse:"

inputs = tokenizer_base(instruction_prompt, return_tensors="pt")

# Generate
output = model_base.generate(
    **inputs,
    max_new_tokens=48,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer_base.eos_token_id
)

print(f"--Prompt--\n{instruction_prompt}\n\n")
print(f"--Generated Output--\n{tokenizer_base.decode(output[0], skip_special_tokens=True)}")



--Prompt--
Instruction: What’s the best way to lose weight quickly?. 
Response:


--Generated Output--
Instruction: What’s the best way to lose weight quickly?. 
Response: The following is very important (thank you!)....
The most important thing is to eat well. 
The most important thing is to eat well. But if you are not able to eat well, then you won't get hungry at


*We often see the model generate nonsense, repeat the word 'Response', or start a new instruction instead of answering the list."*

In [ ]:
# Clean up
del tokenizer_base, model_base, inputs, output
clean_memory()

# **Instruction Fine-Tuning (IFT)**



**Theory:**
To fix the issue above, we perform **Instruction Tuning**.
* **Data:** Few (Task/Instruction, Output) examples.
* **Method:** We fine-tune the pre-trained model on this supervised dataset.

**Architectures for IFT:**
We discussed **T5 (Text-to-Text Transfer Transformer)** and **FLAN** (Fine-tuned Language Net). T5 treats every problem as a text-generation problem ("translate English to German: ...", "summarize: ...").


### **Preparing Instruction Data**
We will simulate a tiny "Instruction Dataset" similar to the **Self-Instruct** format or **FLAN** templates mentioned in slides 28 & 30.

In [ ]:
# In a real IFT setting, this would be thousands of rows loaded from JSONL files.

data_samples = [
    {
        "instruction": "Translate the following English text to French.",
        "input": "Hello, how are you?",
        "output": "Bonjour, comment allez-vous?"
    },
    {
        "instruction": "Summarize the sentence.",
        "input": "The quick brown fox jumps over the lazy dog.",
        "output": "A fox jumped over a dog."
    },
    {
        "instruction": "Classify the sentiment.",
        "input": "I absolutely loved this movie!",
        "output": "Positive"
    },
    {
        "instruction": "Answer the question.",
        "input": "Which character was played by SRK in DDLJ?",
        "output": "Raj."
    },
    {
        "instruction": "Translate to French.",
        "input": "Good morning.",
        "output": "Bonjour."
    },
    {
        "instruction": "Sentiment analysis.",
        "input": "The food was terrible.",
        "output": "Negative"
    }
]

# Convert to Hugging Face Dataset object
dataset = Dataset.from_pandas(pd.DataFrame(data_samples))

print("--- Sample Data Point ---")
print(dataset[0])

--- Sample Data Point ---
{'instruction': 'Translate the following English text to French.', 'input': 'Hello, how are you?', 'output': 'Bonjour, comment allez-vous?'}


### **Fine-Tuning T5 (Text-to-Text)**

We will use **T5-Small**. T5 is an encoder-decoder model that was pre-trained with a span-corruption objective (similar to BART). We will now fine-tune it to follow specific instructions.

Model used: https://huggingface.co/docs/transformers/en/model_doc/t5

In [ ]:
# T5 expects input format: "instruction: [instruction] input: [input]"
# and target: "[output]"

model_checkpoint = "t5-small"
tokenizer_t5 = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # Construct inputs based on T5 text-to-text format
    # We combine instruction and input into a single string
    inputs = [f"{instr} \nInput: {inp}" for instr, inp in zip(examples["instruction"], examples["input"])]
    targets = examples["output"]

    # Tokenize inputs
    model_inputs = tokenizer_t5(inputs, max_length=128, truncation=True)

    # Tokenize targets (labels)
    with tokenizer_t5.as_target_tokenizer():
        labels = tokenizer_t5(targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Split for demo purposes (usually you have separate files)

tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Data Collator handles dynamic padding of batches
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_t5, model=model_t5)


training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=4,
    predict_with_generate=True,
    logging_steps=1,
    use_cpu=not torch.cuda.is_available()
)

trainer = Seq2SeqTrainer(
    model=model_t5,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer_t5,
    data_collator=data_collator,
)

print("Starting Instruction Fine-Tuning...")
trainer.train()
print("Training Complete.")

/tmp/ipython-input-2318254361.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Instruction Fine-Tuning...


Step,Training Loss
1,2.394100
2,3.495400
3,5.745300
4,3.165900
5,1.417900
6,4.667900
7,1.694100
8,2.860600
9,5.099800
10,0.935500


Training Complete.


### **Inference with the Tuned Model**

Now that the model has seen examples of (Instruction -> Output), it should be better at following the format.

In [ ]:
# Let's try a NEW input not in the training set
test_instruction = "Translate the following English text to French."
test_input = "Hello, how are you?"
full_prompt = f"{test_instruction} \nInput: {test_input}"

# Tokenize
inputs = tokenizer_t5(full_prompt, return_tensors="pt").to(model_t5.device)

# Generate
with torch.no_grad():
    outputs = model_t5.generate(**inputs, max_new_tokens=50)

print(f"--- IFT Result ---")
print(f"Prompt: {full_prompt}")
print(f"Generated Output: {tokenizer_t5.decode(outputs[0], skip_special_tokens=True)}")

--- IFT Result ---
Prompt: Translate the following English text to French. 
Input: Hello, how are you?
Generated Output: Bonjour, comment allez-vous?


In [ ]:
# Clean up
del model_t5, tokenizer_t5, trainer
clean_memory()

# **Synthetic Data Generation (Self-Instruct)**

**Theory:**
Where do we get the thousands of instruction pairs?
One popular method is **Self-Instruct**. We use a powerful LLM (like GPT-4) to *generate* instructions for us.

**Process:**
1.  Start with Seed Tasks (Human written).
2.  Prompt LLM: "Come up with a series of tasks..."
3.  Filter for quality.
4.  Add to dataset.

In [ ]:
# Prompt Generation

def generate_synthetic_instruction_prompt(seed_instruction):
    """
    Constructs the meta-prompt used in Self-Instruct to generate new data.
    """
    meta_prompt = f"""
    You are an expert data generator.

    Here is an example instruction:
    "{seed_instruction}"

    Please generate 3 NEW, distinct instructions that are similar in style but different in topic.
    Format them as a numbered list.
    """
    return meta_prompt

seed = "Write a summary of the news article provided below."
final_prompt = generate_synthetic_instruction_prompt(seed)

print("--- Meta-Prompt for Synthetic Data Generation ---")
print(final_prompt)

--- Meta-Prompt for Synthetic Data Generation ---

    You are an expert data generator.

    Here is an example instruction:
    "Write a summary of the news article provided below."

    Please generate 3 NEW, distinct instructions that are similar in style but different in topic.
    Format them as a numbered list.
    


## Summary & Next Steps

1.  **Pre-training:** Teaches the model the "structure" of language via MLM (BERT) or Next Token Prediction (GPT).
2.  **The Gap:** Pre-trained models are good at completion, bad at following orders.
3.  **Instruction Tuning:** We convert tasks into (Instruction, Input, Output) pairs (like FLAN) and fine-tune.
4.  **Data:** We often use Synthetic Data (Self-Instruct, Evol-Instruct) to scale this up cheaply.